# Football Prediction System - Model Performance Analysis

Visualizes the performance of all trained models.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import json, os, joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import LEAGUE_CONFIG
# Find project root by looking for src/ directory
_cwd = os.getcwd()
PROJECT_ROOT = _cwd if os.path.exists(os.path.join(_cwd, 'src')) else os.path.dirname(_cwd)
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, 'outputs')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
print('Libraries loaded!')

In [ ]:
LEAGUES = list(LEAGUE_CONFIG.keys())
rows = []

for lg in LEAGUES:
    league_dir = os.path.join(MODELS_DIR, lg)
    if not os.path.exists(league_dir):
        continue
    for f in os.listdir(league_dir):
        if not f.endswith('.pkl'):
            continue
        basename = f.replace('.pkl', '')
        model_type = None
        market = None
        for mt in ['gradient_boosting', 'random_forest', 'xgboost']:
            if basename.endswith('_' + mt):
                model_type = mt
                market = basename[:-len(mt)-1]
                break
        if model_type is None:
            continue
        try:
            data = joblib.load(os.path.join(league_dir, f))
            acc = data.get('results', {}).get('accuracy')
            mae = data.get('results', {}).get('mae')
            rows.append({
                'League': LEAGUE_CONFIG[lg]['name'],
                'Market': market,
                'Model': model_type,
                'Accuracy': acc,
                'MAE': mae
            })
        except Exception as e:
            print(f'Error loading {f}: {e}')

models_df = pd.DataFrame(rows)
print(f'Loaded {len(models_df)} trained models')
models_df.head(10)

## 2. Best Model per Market per League

In [ ]:
class_df = models_df[models_df['Accuracy'].notna()].copy()

if not class_df.empty:
    best = class_df.loc[class_df.groupby(['League', 'Market'])['Accuracy'].idxmax()]
    summary = best[['League', 'Market', 'Model', 'Accuracy']].sort_values(['League', 'Accuracy'], ascending=[True, False])
    print('Best Model per Market per League:')
    print('=' * 70)
    for league in summary['League'].unique():
        print(f'\n{league}:')
        for _, row in summary[summary['League'] == league].iterrows():
            print(f"  {row['Market']:25s}: {row['Accuracy']:.4f} ({row['Model']})")
else:
    print('No classification models found')

## 3. Accuracy Heatmap

In [ ]:
if not class_df.empty:
    pivot = best.pivot_table(values='Accuracy', index='League', columns='Market', aggfunc='first')
    pivot = pivot.apply(pd.to_numeric, errors='coerce')
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5, ax=ax, vmin=0.3, vmax=0.75)
    ax.set_title('Best Model Accuracy by League and Market')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, '02_accuracy_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/02_accuracy_heatmap.png')

## 4. Model Type Comparison

In [ ]:
if not class_df.empty:
    key_markets = ['match_result', 'over_under_25', 'btts']
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for idx, market in enumerate(key_markets):
        md = class_df[class_df['Market'] == market].copy()
        if not md.empty:
            pvt = md.pivot_table(values='Accuracy', index='League', columns='Model', aggfunc='first')
            pvt.plot(kind='bar', ax=axes[idx], edgecolor='black')
            axes[idx].set_title(market.replace('_', ' ').title())
            axes[idx].set_ylabel('Accuracy')
            axes[idx].set_ylim(0.3, 0.65)
            plt.setp(axes[idx].get_xticklabels(), rotation=45, ha='right')
    plt.suptitle('Model Type Comparison by Market', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, '02_model_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/02_model_comparison.png')

## 5. Market Difficulty Ranking

In [ ]:
if not class_df.empty:
    avg_by_market = class_df.groupby('Market')['Accuracy'].agg(['mean', 'std', 'min', 'max']).sort_values('mean', ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#2ecc71' if x > 0.55 else '#f39c12' if x > 0.45 else '#e74c3c' for x in avg_by_market['mean']]
    avg_by_market['mean'].plot(kind='barh', xerr=avg_by_market['std'], ax=ax, color=colors, edgecolor='black', capsize=3)
    ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('Average Accuracy by Market (across all leagues)')
    for i, (idx_name, row) in enumerate(avg_by_market.iterrows()):
        ax.text(row['mean'] + 0.01, i, f"{row['mean']:.3f}", va='center')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, '02_market_ranking.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/02_market_ranking.png')
    print('\nMarket Ranking:')
    print(avg_by_market.round(4))

## 6. League Difficulty Ranking

In [ ]:
if not class_df.empty:
    avg_by_league = class_df.groupby('League')['Accuracy'].agg(['mean', 'std']).sort_values('mean', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#2ecc71' if x > 0.50 else '#f39c12' if x > 0.45 else '#e74c3c' for x in avg_by_league['mean']]
    avg_by_league['mean'].plot(kind='barh', xerr=avg_by_league['std'], ax=ax, color=colors, edgecolor='black', capsize=3)
    ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('Average Model Accuracy by League')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, '02_league_ranking.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/02_league_ranking.png')
    print('\nLeague Ranking:')
    print(avg_by_league.round(4))

In [ ]:
models_df.to_csv(os.path.join(OUTPUTS_DIR, 'model_performance_summary.csv'), index=False)
print(f'Saved to {OUTPUTS_DIR}/model_performance_summary.csv')